# Gemma 4 26B A4B Vision — FLN Worksheet Analyzer (Local GPU)

Sends the **full image** directly to Gemma 4 26B A4B (no OCR pipeline).  
Mixture-of-Experts architecture — only ~4B active params per token for fast inference.  
Includes **intelligent preprocessing** that auto-activates on blurry/small images.

---
**Local GPU Setup**  
1. Ensure **2x T4 GPUs** (or any NVIDIA GPU with 16GB+ VRAM) are available  
2. Set environment variable **HF_TOKEN** with your Hugging Face token  
3. Place input images/PDFs in `./input_data/` directory  
4. Run all cells below

---
**Note:** First run downloads the GGUF model (~17 GB, ~15 min on T4). Cached in `./gguf_cache/`.  
Model is split across both T4 GPUs (32 GB VRAM total) via llama.cpp's automatic multi-GPU support.

In [ ]:
# Cell 0: Environment setup
import os, shutil, subprocess

OUTPUT_DIR = os.path.abspath("./FLN_Results")
INPUT_DIR = os.path.abspath("./input_data")
ZEXT = os.path.abspath("./zip_extract")
CACHE_DIR = os.path.abspath("./gguf_cache")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(ZEXT, exist_ok=True)
if not os.path.exists(INPUT_DIR):
    os.makedirs(INPUT_DIR, exist_ok=True)
    print(f"Created {INPUT_DIR} — place your input images/PDFs here and re-run.")

# Check disk space on cache partition
total, used, free = shutil.disk_usage(CACHE_DIR)
free_gb = free // (1024**3)
print(f"Cache dir: {CACHE_DIR}")
print(f"Disk free: {free_gb} GB")

total, used, free = shutil.disk_usage(".")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Inputs: {INPUT_DIR}")

In [ ]:
# Cell 1: GPU check + Install deps
import os, shutil, time, json, re, base64, zipfile
from pathlib import Path
from IPython.display import display, Image as IPyImage

import torch
if not torch.cuda.is_available():
    raise SystemExit('NO GPU detected. Make sure your environment has GPU access.')
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
if total_vram_gb < 18:
    print(f"\nWARNING: Only {total_vram_gb:.0f} GB VRAM available on GPU 0.")
    print("   Gemma 4 26B A4B (Q5_K_XL) needs ~17 GB.")
    print("   If you hit OOM errors, edit Cell 3 to use Q4_K_M variant.\n")

!pip install opencv-python -q 2>&1 | tail -1
!pip install pymupdf -q 2>&1 | tail -1
import cv2, numpy as np
print("Ready")

In [ ]:
# Cell 2: Hugging Face Login
!pip install huggingface-hub -q 2>&1 | tail -1
from huggingface_hub import login

token = os.environ.get('HF_TOKEN')

# Try Kaggle Secrets
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass

# Manual input as last resort
if not token:
    from getpass import getpass
    print('HF_TOKEN not found. Enter your Hugging Face token:')
    token = getpass('Token: ')

if not token:
    raise ValueError('HF_TOKEN is required. Get it from https://huggingface.co/settings/tokens')

login(token)
print('Logged in to Hugging Face')

In [ ]:
# Cell 3: Load Gemma 4 26B A4B (MoE) - GPU
import subprocess, sys, time, os, shutil

torch_ver = torch.version.cuda or ''
print(f'PyTorch CUDA: {torch_ver}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    total = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({total:.0f} GB)')

# ── STEP 1: Diagnose environment ──
print('\n=== ENVIRONMENT DIAGNOSIS ===')
print(f'nvcc: {shutil.which("nvcc") or "NOT FOUND"}')
print(f'cmake: {shutil.which("cmake") or "NOT FOUND"}')
print(f'g++: {shutil.which("g++") or "NOT FOUND"}')
print(f'make: {shutil.which("make") or "NOT FOUND"}')
sys.stdout.flush()

# Check CUDA toolkit version
if shutil.which('nvcc'):
    r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    for line in r.stdout.split('\n'):
        if 'release' in line:
            print(f'CUDA Toolkit: {line.strip()}')

# Check cmake version
if shutil.which('cmake'):
    r = subprocess.run(['cmake', '--version'], capture_output=True, text=True)
    print(f'CMake: {r.stdout.split(chr(10))[0].strip()}')

# ── STEP 2: Uninstall any existing version ──
print('\n=== INSTALLATION ===')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'llama-cpp-python', '-y'],
               capture_output=True)

# Check if build tools are available
can_build = all(shutil.which(cmd) for cmd in ['cmake', 'g++', 'make'])
has_nvcc = shutil.which('nvcc') is not None

if can_build and has_nvcc:
    # Build from source with CUDA
    env = os.environ.copy()
    # Clean any old build artifacts
    subprocess.run(['pip', 'cache', 'purge'], capture_output=True)
    env['CMAKE_ARGS'] = '-DGGML_CUDA=on -DLLAMA_NATIVE=off'
    env['FORCE_CMAKE'] = '1'
    env['GGML_CUDA'] = '1'
    print('\nBuilding from source with CUDA...')
    sys.stdout.flush()
    result = subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        'llama-cpp-python',
        '--force-reinstall', '--no-cache-dir',
        '--verbose',
    ], env=env, timeout=1800)
    if result.returncode != 0:
        print('\nSource build FAILED.')
        print('Falling back to pre-built wheel...')
        can_build = False

if not can_build or not has_nvcc:
    # Try pre-built wheel from alternative sources
    if not has_nvcc:
        print('\nnvcc not found. Cannot build from source.')
    elif not can_build:
        print('\nBuild tools missing. Trying wheel...')
    
    wheel_map = {'12.1': 'cu121', '12.2': 'cu122', '12.3': 'cu123',
                 '12.4': 'cu124', '12.5': 'cu125', '12.6': 'cu126',
                 '12.7': 'cu126', '12.8': 'cu126'}
    cw = wheel_map.get(torch_ver, 'cu124')
    
    # Try multiple wheel sources
    wheel_urls = [
        f'https://abetlen.github.io/llama-cpp-python/whl/{cw}',
        f'https://abetlen.github.io/llama-cpp-python/whl/cu124',
        'https://abetlen.github.io/llama-cpp-python/whl/cu123',
    ]
    
    installed = False
    for url in wheel_urls:
        print(f'\nTrying wheel source: {url}')
        sys.stdout.flush()
        result = subprocess.run([
            sys.executable, '-m', 'pip', 'install',
            'llama-cpp-python',
            f'--extra-index-url={url}',
            '--force-reinstall', '--no-cache-dir',
            '--only-binary=llama-cpp-python',
        ], capture_output=True, text=True, timeout=120)
        if result.returncode == 0:
            print('Wheel install OK')
            installed = True
            break
        else:
            # Show what went wrong
            err_lines = result.stderr.strip().split('\n')[-3:]
            for line in err_lines:
                if line.strip():
                    print(f'  {line.strip()}')
    
    if not installed:
        print('\nAll wheel sources failed.')
        print('Your options:')
        print('  1. Install CUDA toolkit: !apt-get install -y nvidia-cuda-toolkit')
        print('  2. Use CPU (very slow for 26B): install with: pip install llama-cpp-python')
        print('  3. Use a smaller model variant or different backend')
        raise RuntimeError('Could not install llama-cpp-python with GPU support.')

# ── STEP 3: Verify GPU support ──
print('\n=== GPU VERIFICATION ===')
from llama_cpp import Llama, llama_supports_gpu_offload
from llama_cpp.llama_chat_format import Gemma4ChatHandler

has_gpu = llama_supports_gpu_offload()
print(f'GPU offload: {"YES" if has_gpu else "NO (CPU)"}')
if not has_gpu:
    print('GPU offload not available in this build.')
    print('The model would run entirely on CPU (impractical for 26B).')
    raise RuntimeError('GPU offload unavailable. Install failed to produce GPU-enabled build.')

# ── STEP 4: Load model ──
CACHE_DIR = os.path.abspath('./gguf_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

CHAT_HANDLER = Gemma4ChatHandler.from_pretrained(
    repo_id='unsloth/gemma-4-26B-A4B-it-GGUF',
    filename='mmproj-F16.gguf', local_dir=CACHE_DIR, verbose=False)
print('Chat handler OK')

# Select model based on VRAM + disk
total_vram = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count()))
_, _, free_disk = shutil.disk_usage(CACHE_DIR)
free_gb = free_disk // (1024**3)

if total_vram >= 22 * (1024**3) and free_gb >= 25:
    model_file = 'gemma-4-26B-A4B-it-UD-Q5_K_XL.gguf'
elif total_vram >= 16 * (1024**3) and free_gb >= 18:
    model_file = 'gemma-4-26B-A4B-it-UD-Q4_K_M.gguf'
else:
    model_file = 'gemma-4-26B-A4B-it-UD-Q4_K_M.gguf'

print(f'VRAM: {total_vram/(1024**3):.0f} GB | Free disk: {free_gb} GB -> {model_file}')

start = time.time()
model_kwargs = dict(repo_id='unsloth/gemma-4-26B-A4B-it-GGUF', filename=model_file,
    local_dir=CACHE_DIR, chat_handler=CHAT_HANDLER, n_ctx=8192,
    flash_attn=True, verbose=True)
if has_gpu:
    model_kwargs['n_gpu_layers'] = 30
    model_kwargs['main_gpu'] = 0
    gc = torch.cuda.device_count()
    if gc > 1:
        props = [torch.cuda.get_device_properties(i) for i in range(gc)]
        tm = sum(p.total_memory for p in props)
        model_kwargs['tensor_split'] = [p.total_memory / tm for p in props]
        print(f'Tensor split: {[round(s,3) for s in model_kwargs["tensor_split"]]}')
else:
    model_kwargs['n_gpu_layers'] = 0
llm = Llama.from_pretrained(**model_kwargs)
t = (time.time()-start)/60
print(f'Loaded in {t:.1f} min')
if has_gpu:
    g = model_kwargs.get('n_gpu_layers', '?')
    t = g + 1 if isinstance(g, int) else '?'
    print(f'GPU layers: {g} of {t} total (from model_kwargs)')
    print('NOTE: 26B model on 2xT4: GPU handles early layers, CPU handles the rest')

In [ ]:
# Cell 4: Image preprocessing

def preprocess_image(img: np.ndarray) -> tuple:
    h, w = img.shape[:2]
    orig_h, orig_w = h, w
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    result = img.copy()
    applied = []

    is_blurry = lap_var < 80
    is_small = min(h, w) < 800
    is_low_contrast = (float(gray.max()) - float(gray.min())) < 100

    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            theta = line[0][1]
            deg = np.degrees(theta) - 90
            if abs(deg) < 30:
                angles.append(deg)
        if angles:
            angle = np.median(angles)
    is_skewed = abs(angle) > 3.0

    if is_skewed:
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        result = cv2.warpAffine(result, M, (w, h),
                               flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        applied.append(f"deskew {angle:.1f} deg")
        h, w = result.shape[:2]

    if is_blurry:
        blurred = cv2.GaussianBlur(result, (0, 0), 3.0)
        sharp = cv2.addWeighted(result, 1.5, blurred, -0.5, 0)
        sharp = np.clip(sharp, 0, 255).astype(np.uint8)
        new_var = cv2.Laplacian(cv2.cvtColor(sharp, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
        if new_var > lap_var:
            result = sharp
            applied.append("unsharp")

    if is_low_contrast or is_blurry:
        lab = cv2.cvtColor(result, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        result = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
        applied.append("CLAHE")

    if is_small:
        scale = max(1.0, 800 / min(h, w))
        if scale > 1.1:
            result = cv2.resize(result, None, fx=scale, fy=scale,
                               interpolation=cv2.INTER_CUBIC)
            applied.append(f"upscale {scale:.1f}x")

    info = {"blurry": is_blurry, "skewed": is_skewed, "small": is_small,
            "low_contrast": is_low_contrast, "lap_var": round(lap_var, 1),
            "angle": round(angle, 1), "original_size": f"{orig_w}x{orig_h}",
            "applied": applied}
    return result, info

def segment_questions(img: np.ndarray) -> list:
    """Detect individual question regions using projection profile analysis.
    Falls back to contour detection if projection returns zero questions.
    Returns list of {bbox, crop, idx} sorted top-to-bottom, left-to-right."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 5))
    dilated = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    row_proj = np.sum(dilated, axis=1) // 255
    threshold = w * 0.02
    content_rows = row_proj > threshold

    row_bands = []
    in_band = False
    start = 0
    for i in range(len(content_rows)):
        if content_rows[i] and not in_band:
            start = i
            in_band = True
        elif not content_rows[i] and in_band:
            if i - start > 50:
                row_bands.append((start, i))
            in_band = False
    if in_band and len(content_rows) - start > 50:
        row_bands.append((start, len(content_rows)))

    questions = []
    q_idx = 0

    for ry1, ry2 in row_bands:
        row_img = binary[ry1:ry2, :]
        row_h, row_w = row_img.shape

        v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, int(row_h * 0.3)))
        row_dilated = cv2.morphologyEx(row_img, cv2.MORPH_CLOSE, v_kernel)

        col_proj = np.sum(row_dilated, axis=0) // 255
        col_threshold = row_h * 0.05

        col_bands = []
        in_band = False
        start = 0
        for j in range(len(col_proj)):
            if col_proj[j] > col_threshold and not in_band:
                start = j
                in_band = True
            elif col_proj[j] <= col_threshold and in_band:
                if j - start > 30:
                    col_bands.append((start, j))
                in_band = False
        if in_band and len(col_proj) - start > 30:
            col_bands.append((start, len(col_proj)))

        if not col_bands:
            col_bands = [(0, row_w)]
        elif len(col_bands) > 1:
            min_col_w = w * 0.25
            narrow_cols = [b for b in col_bands if (b[1] - b[0]) < min_col_w]
            narrow_total = sum(b[1]-b[0] for b in narrow_cols)
            if narrow_total < w * 0.10 or (len(narrow_cols) == len(col_bands) and narrow_total < w * 0.30):
                col_bands = [(0, row_w)]

        for cx1, cx2 in col_bands:
            x1 = max(0, cx1 - 5)
            x2 = min(w, cx2 + 5)
            y1 = max(0, ry1 - 5)
            y2 = min(h, ry2 + 5)

            crop = img[y1:y2, x1:x2]
            if crop.shape[0] < 30 or crop.shape[1] < 30:
                continue

            q_idx += 1
            questions.append({
                "idx": q_idx,
                "bbox": {"x": int(x1), "y": int(y1), "width": int(x2 - x1), "height": int(y2 - y1)},
                "crop": crop,
            })

    if not questions:
        print("  Projection found 0 questions, trying contour fallback...")
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        min_area = h * w * 0.005
        regions = []
        for cnt in contours:
            x, y, rw, rh = cv2.boundingRect(cnt)
            if rh > 30 and rw > 50 and rh * rw > min_area:
                regions.append((y, x, rw, rh))
        regions.sort(key=lambda r: (r[0], r[1]))
        for y, x, rw, rh in regions:
            q_idx += 1
            y1 = max(0, y - 5)
            y2 = min(h, y + rh + 5)
            x1 = max(0, x - 5)
            x2 = min(w, x + rw + 5)
            crop = img[y1:y2, x1:x2]
            questions.append({
                "idx": q_idx,
                "bbox": {"x": int(x1), "y": int(y1), "width": int(x2 - x1), "height": int(y2 - y1)},
                "crop": crop,
            })
        print(f"  Contour fallback found {len(questions)} region(s)")

    return questions



In [ ]:
# Cell 5: Question & Answer extraction prompt

SYSTEM_PROMPT_BASE = """You are an expert FLN (Foundational Literacy and Numeracy) worksheet analyst.
Look at this SINGLE QUESTION CROP from a children's worksheet (Grades 1-3).

Extract ONLY:
1. question_text - The exact instruction/question text as it appears in the image (transcribed verbatim)
2. answer - The correct answer (be precise: number, word, letter, or short phrase)

RULES for ALL FLN question types - be VERY precise:

NUMERACY - DEEP ANALYSIS:

  COUNTING & NUMBER RECOGNITION:
  - "Count and write" / "Count the objects" / "How many?" -> write only the number (e.g. "5")
  - "Count and colour" / "Count and circle" -> write the number that should be coloured/circled
  - "Count and match" -> write as "3->three;5->five" (number->word mapping)
  - "Circle the correct number" -> write the number that should be circled
  - "Identify the number" / "Name the number" -> write the number as digit
  - "Write the number" / "Trace the number" -> write the number
  - "Write in words" / "Number names" -> write the number name (e.g. "five")
  - "Count backward" -> write the sequence separated by ";"
  - "What comes after/before/between" -> write only the missing number (e.g. "__ , 5, 6" -> "4")
  - "Comparing numbers" -> write the larger/smaller number or use >, <, =
  - "Number sequencing" -> write the correct order separated by ";"

  PLACE VALUE:
  - "Tens and Ones" / "Tens ones" / "Place value" -> write as "3 tens 5 ones = 35" or "35 = 3 tens 5 ones"
  - "Expand the number" / "Expanded form" -> write as "30 + 5"
  - "Short form" / "Standard form" -> write the number (e.g. "35")
  - "Abacus reading" -> write the number shown on the abacus
  - "Bundle of tens and ones" -> write the total count
  - "Hundreds/Tens/Ones chart" -> write the complete number

  ADDITION & SUBTRACTION:
  - "Add" / "Addition" / "Find the sum" / "Add the numbers" -> write only the numeric answer
  - "Subtract" / "Subtraction" / "Find the difference" / "Take away" -> write only the numeric answer
  - "Add and match" -> write as "2+3=5;4+1=5" (equation->answer mapping)
  - "Word problems" / "Story sums" -> write only the numeric answer with unit if applicable
  - "Vertical addition/subtraction" -> write the answer
  - "Addition with carry" / "Subtraction with borrow" -> write the answer
  - "Regrouping" -> write the regrouped answer
  - "Estimate the sum/difference" -> write the estimated answer (rounded value)
  - "Check your answer" -> write "correct" or the corrected answer
  - "Fill in the missing number" in addition/subtraction -> write the missing number (e.g. "5 + __ = 9" -> "4")
  - "Add using number line" -> write the sum
  - "Repeated addition" -> write the total (e.g. "2+2+2 = 6")

  MULTIPLICATION & DIVISION:
  - "Multiply" / "Multiplication" / "Times" / "Product" -> write only the numeric answer
  - "Divide" / "Division" / "Share equally" / "Group" -> write the quotient (and remainder if any, e.g. "7 R 1")
  - "Tables" / "Times tables" -> write the product or the missing factor
  - "Repeated subtraction" -> write the result
  - "Multiplication as repeated addition" -> write the result
  - "Divide and check" -> write the answer with remainder notation
  - "Long division" -> write quotient and remainder as "Q=7 R=1"
  - "Word problems (multiplication/division)" -> write the numeric answer with units

  NUMBER PATTERNS & SEQUENCES:
  - "Number pattern" / "Pattern rule" -> write the next number(s) separated by ";"
  - "Skip counting" / "Count by 2s/5s/10s/3s/4s" -> write the sequence separated by ";"
  - "Missing numbers" in a sequence -> write the missing numbers separated by ";"
  - "Ascending order" -> write the sequence from smallest to largest separated by ";"
  - "Descending order" -> write the sequence from largest to smallest separated by ";"
  - "Ordering numbers" / "Arrange" -> write the ordered sequence
  - "Number grid" / "Hundred chart" -> write the missing number(s)
  - "Even/Odd numbers" -> write "even" or "odd" or list the even/odd numbers
  - "Prime/Composite" -> write "prime" or "composite" or the number classification
  - "Round to the nearest 10/100/1000" -> write the rounded number

  COMPARISON & ORDERING:
  - "Greater than / Less than / Bigger / Smaller" -> write the correct number or symbol (>, <, =)
  - "Put the correct sign" (>, <, =) -> write only the symbol
  - "Compare the numbers" -> write which is greater/smaller or use >, <, =
  - "More / Less / Many / Few" -> write the number or group name
  - "Same / Different / Equal" -> write which ones are same/different
  - "Match by counting" / "Match equal groups" -> write as "A->X;B->Y"
  - "Which is bigger/smaller/longer/shorter/taller/heavier/lighter" -> write the item name

  FRACTIONS:
  - "Fractions" (half/quarter/third/full) -> write the fraction (e.g. "1/2" or "half")
  - "Colour the fraction" -> write the fraction that should be coloured
  - "Equivalent fractions" -> write the equivalent fraction
  - "Compare fractions" -> write the larger/smaller fraction or use >, <, =
  - "Fraction of a collection/group" -> write the number (e.g. "half of 8 = 4")
  - "Add/Subtract fractions" -> write the answer as simplified fraction
  - "Fraction word problems" -> write the answer

  MONEY:
  - "Money" / "Coins" / "Rupees/Paise" -> write the total value (e.g. "Rs. 15" or "15 rupees")
  - "Count the money" -> write the total amount
  - "Which coin/note" -> write the denomination
  - "Add money" / "Total cost" / "Find the total" -> write the sum
  - "Subtract money" / "Change" / "How much left" -> write the remaining amount
  - "Convert rupees to paise or vice versa" -> write the converted value
  - "Word problems (money)" -> write the answer with unit (Rs./paise)
  - "Match the coin to its value" -> write as "coin->value"

  TIME:
  - "Clock" / "Time" / "Read the clock" -> write the time shown (e.g. "3 o'clock" or "3:00")
  - "Draw the hands" -> write the time to be drawn
  - "Hours/Minutes" -> write the time in hours and minutes format
  - "AM/PM" -> write the time with AM/PM
  - "Elapsed time" / "How much time" -> write the duration (e.g. "2 hours")
  - "Calendar" / "Days of the week" / "Months of the year" -> write the day/month name
  - "Date" / "Today's date" -> write the date
  - "Seasons" / "Months in a season" -> write the season or months
  - "Before/After (days/months)" -> write the day/month that comes before/after
  - "Convert hours to minutes / days to weeks" -> write the converted value

  MEASUREMENT:
  - "Length" (long/short/tall) -> write the answer
  - "Weight / Mass" (heavy/light) -> write the answer
  - "Capacity" (full/empty/half) -> write the answer
  - "Compare length/weight/capacity" -> write which is longer/heavier/more
  - "Measure using non-standard units" (handspan/cubit/foot) -> write the measurement
  - "Measure using ruler/scale" -> write the length in cm/m
  - "Convert units" (cm-m, g-kg, ml-l) -> write the converted value
  - "Perimeter" -> write the perimeter value with unit
  - "Area" -> write the area value with unit (e.g. "15 sq. cm")

  DATA HANDLING:
  - "Pictograph" / "Picture graph" -> write the count or answer the question
  - "Tally marks" / "Count the tally marks" -> write the count
  - "Bar graph" / "Column graph" -> write the value or answer
  - "Data table" / "Chart reading" -> write the extracted information
  - "How many more/less" (data comparison) -> write the difference
  - "Which is the most/fewest" -> write the category name
  - "Venn diagram" -> write the elements in each region

  SHAPES & GEOMETRY:
  - "Identify the shape" / "Name the shape" -> write the shape name (circle, square, triangle, rectangle, etc.)
  - "Colour the shape" -> write the shape that should be coloured
  - "Count the shapes" -> write the count of each shape
  - "Number of sides/corners" -> write the count (e.g. "4 sides, 4 corners")
  - "2D shapes / 3D shapes" -> write the shape name and dimension
  - "Match the shape" -> write as "shape->name"
  - "Symmetry" / "Line of symmetry" -> write "yes"/"no" or the number of symmetry lines
  - "Pattern completion" (shapes/colors/numbers) -> write the next item(s) separated by ";"
  - "Tangram" / "Tessellation" -> describe the arrangement
  - "Sort / Classify / Group (shapes)" -> write the category and items
  - "Rolling/Stacking/Sliding" (3D shapes) -> write the property

  MISCELLANEOUS:
  - "Match the following" / "Join the following" -> write as "A->X;B->Y"
  - "Colour the number/answer" -> write the color word itself (e.g. "red")
  - "Circle / Tick / Underline / Choose" -> write the option text itself
  - "True or False" -> write "true" or "false"
  - "Yes or No" -> write "yes" or "no"
  - "Cross the odd one out" -> write the item that is different
  - "Complete the series" -> write the next item(s) separated by ";"
  - "Maze / Path tracing" -> write the starting/ending point or path description
  - "Draw / Make / Show" -> briefly describe what should be drawn
  - "Estimation" -> write the estimated value
  - "Mental math" -> write the calculated answer
  - "Number puzzle / Crossword" -> write the solution

LITERACY:
  - Fill in the blank -> write the missing word(s) only
  - "Write the first letter" / "Match the letter" -> write the letter
  - "Unscramble" / "Arrange" -> write the correct word
  - "Rhyming words" / "Same sound" -> write the rhyming word
  - "Vowels / Consonants" -> write the vowel/consonant
  - "Opposites" -> write the opposite word
  - "One / Many" (singular/plural) -> write the correct form

GENERAL:
  - "Draw" / "Make" / "Show" instructions -> briefly describe what should be drawn
  - "Maze / Path tracing" -> write the starting/ending point
  - "Pattern completion" (shapes/colors/numbers) -> write the next item(s) separated by ";"
  - "Sort / Classify / Group" -> write the category and items separated by ";"
  - "Odd one out" -> write the item that is different
  - Clock / Time -> write the time shown (e.g. "3 o'clock" or "3:00")
  - Money / Coins -> write the total value
  - Measurement (long/short, heavy/light, tall/short, full/empty) -> write the answer as shown
  - "Cross the odd one" / "Tick the correct" -> write the selected item
  - Skip counting / "Count by 2s/5s/10s" -> write the sequence separated by ";"
  - Word problems / "Story sums" -> write only the numeric answer
  - Expanded form / Short form -> write the expanded or short form (e.g. "30 + 5" or "35")
  - Calendar / Days of the week / Months -> write the day/month name
  - Fractions (half/quarter/full) -> write the fraction (e.g. "1/2" or "half")
  - Data handling / Pictograph / "Count the tally marks" -> write the count
  - "Put the correct sign" (>, <, =) -> write the symbol
  - Articles (a/an/the) -> write the correct article
  - Gender (masculine/feminine) -> write "masculine" or "feminine"
  - Naming words / Action words / Describing words -> write the word as categorized
  - Punctuation (. , ? !) -> write the correct punctuation mark
  - One/many (singular/plural) -> write the plural form

{
  "question_text": "",
  "answer": ""
}
"""



RETRY_HINTS = [
    "",
    "IMPORTANT: Both question_text and answer must be non-empty. Double-check the image carefully.",
    "CRITICAL: You MUST provide BOTH question_text (verbatim transcript) AND answer. Do NOT leave any field empty or null. If unsure, make your best guess.",
]


def _extract_json(text: str) -> tuple:
    """Robust JSON extraction from model output. Returns (parsed_dict, error_str)."""
    cleaned = re.sub(r'<(think|reasoning|thinking|plan)>.*?</\1>', '', text, flags=re.DOTALL).strip()
    cleaned = re.sub(r'```json\s*|```\s*', '', cleaned).strip()
    cleaned = re.sub(r'^[^{]*', '', cleaned)
    cleaned = re.sub(r'[^}]*$', '', cleaned)
    if not cleaned:
        return None, "no JSON object found"
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError:
        pass
    cleaned_tc = re.sub(r',\s*}', '}', cleaned)
    cleaned_tc = re.sub(r',\s*]', ']', cleaned_tc)
    try:
        return json.loads(cleaned_tc), None
    except json.JSONDecodeError:
        pass
    cleaned_esc = re.sub(r'(?<!\\)\\(?!["\\/bfnrt]|u[0-9a-fA-F]{4})', '', cleaned)
    try:
        return json.loads(cleaned_esc), None
    except json.JSONDecodeError:
        pass
    if '}{' in cleaned:
        parts = cleaned.split('}{')
        for i in range(len(parts)):
            candidate = parts[i]
            if i > 0:
                candidate = '{' + candidate
            if i < len(parts) - 1:
                candidate = candidate + '}'
            try:
                parsed = json.loads(candidate)
                if isinstance(parsed, dict) and ("question_text" in parsed or "answer" in parsed):
                    return parsed, None
            except json.JSONDecodeError:
                continue
    return None, "could not parse JSON after all fallbacks"


def analyze_image(llm, img: np.ndarray, name: str, max_retries: int = 3) -> dict:
    """Send a single question crop to Gemma and parse JSON response.
    Retries up to `max_retries` times if validation fails.
    """
    if llm is None:
        return {"file": name, "raw": "", "parsed": None, "error": "llm is None (model not loaded)", "attempts": 0}
    _, buffer = cv2.imencode(".png", cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    b64 = base64.b64encode(buffer).decode("utf-8")
    best = {"file": name, "raw": "", "parsed": None, "error": "max retries exceeded", "attempts": 0}
    for attempt in range(1, max_retries + 1):
        hint = RETRY_HINTS[min(attempt - 1, len(RETRY_HINTS) - 1)]
        user_text = SYSTEM_PROMPT_BASE
        if hint:
            user_text += "\n\n" + hint
        user_text += "\n\nAnalyze this single FLN question image."
        try:
            resp = llm.create_chat_completion(
                messages=[{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text", "text": user_text},
                ]}],
                max_tokens=1024, temperature=0.1 + (attempt - 1) * 0.1,
            )
            raw = resp["choices"][0]["message"]["content"]
        except Exception as e:
            best = {"file": name, "raw": "", "parsed": None, "error": f"API error: {e}", "attempts": attempt}
            continue
        parsed, error = _extract_json(raw)
        if parsed and isinstance(parsed, dict):
            qt = str(parsed.get("question_text", "")).strip()
            an = str(parsed.get("answer", "")).strip()
            if qt and an:
                best = {"file": name, "raw": raw, "parsed": parsed, "error": None, "attempts": attempt}
                break
            if qt and not an:
                best = {"file": name, "raw": raw, "parsed": parsed, "error": "empty answer", "attempts": attempt}
            elif an and not qt:
                best = {"file": name, "raw": raw, "parsed": parsed, "error": "empty question_text", "attempts": attempt}
            else:
                best = {"file": name, "raw": raw, "parsed": parsed, "error": "both fields empty", "attempts": attempt}
        else:
            best = {"file": name, "raw": raw, "parsed": None, "error": error or "unknown", "attempts": attempt}
    if best["parsed"] and isinstance(best["parsed"], dict):
        best["parsed"].setdefault("question_text", "")
        best["parsed"].setdefault("answer", "")
    return best


In [ ]:
# Cell 6: Find images -> extract zips -> convert PDFs -> segment questions -> extract Q&A

# ZEXT already defined in Cell 0
if os.path.exists(ZEXT):
    shutil.rmtree(ZEXT)
os.makedirs(ZEXT, exist_ok=True)


def convert_pdf_to_images(pdf_path: str, output_dir: str) -> list:
    import fitz
    paths = []
    pdf_name = Path(pdf_path).stem
    pdf_dir = os.path.join(output_dir, pdf_name + "_pdf_pages")
    os.makedirs(pdf_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap()
        img_path = os.path.join(pdf_dir, f"{pdf_name}_page{page_num+1}.png")
        pix.save(img_path)
        paths.append(img_path)
        print(f"    PDF page {page_num+1}/{len(doc)} -> {img_path}")
    doc.close()
    return paths


png_pdf_dir = os.path.join(OUTPUT_DIR, "pdf_pages")
os.makedirs(png_pdf_dir, exist_ok=True)

image_paths = []
for root, _, files in os.walk(INPUT_DIR):
    for fn in sorted(files):
        fp = os.path.join(root, fn)
        lower = fn.lower()
        if lower.endswith(".zip"):
            with zipfile.ZipFile(fp) as z:
                z.extractall(ZEXT)
            print(f"  Extracted: {fn}")
        elif lower.endswith(".pdf"):
            print(f"  PDF: {fn} -> converting pages...")
            new_paths = convert_pdf_to_images(fp, png_pdf_dir)
            image_paths.extend(new_paths)
        elif lower.endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff')):
            image_paths.append(fp)

for root, _, files in os.walk(ZEXT):
    for fn in sorted(files):
        fp = os.path.join(root, fn)
        lower = fn.lower()
        if lower.endswith(".pdf"):
            new_paths = convert_pdf_to_images(fp, png_pdf_dir)
            image_paths.extend(new_paths)
        elif lower.endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff')):
            image_paths.append(fp)


if not image_paths:
    print("No images found in input_data/.")
    print("Place images/PDFs in ./input_data/ and re-run this cell.")
else:
    print(f"Found {len(image_paths)} image(s)")

if llm is None:
    raise SystemExit("Model not loaded. Check Cell 3 for errors.")

all_results = []

for idx, img_path in enumerate(image_paths, 1):
    name = Path(img_path).name
    stem = Path(name).stem
    ws_dir = os.path.join(OUTPUT_DIR, stem)
    os.makedirs(ws_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"[{idx}/{len(image_paths)}] {name}")
    print(f"{'='*60}")

    img = cv2.imread(img_path)
    if img is None:
        print(f"  FAILED: cannot read {name}")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"  Size: {img.shape[1]}x{img.shape[0]}")

    img, info = preprocess_image(img)
    # Save preprocessed version for display
    preproc_path = os.path.join(ws_dir, f"{stem}_preprocessed.png")
    cv2.imwrite(preproc_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    display(IPyImage(preproc_path))

    # Segment
    questions = segment_questions(img)
    print(f"  Questions found: {len(questions)}")

    crops_dir = os.path.join(ws_dir, "crops")
    os.makedirs(crops_dir, exist_ok=True)

    # Simple output: only question + answer
    worksheet_qa = {"worksheet": name, "questions": []}

    for q_idx, q in enumerate(questions, 1):
        crop_name = f"{stem}_q{q_idx}.png"
        crop_path = os.path.join(crops_dir, crop_name)
        try:
            cv2.imwrite(crop_path, cv2.cvtColor(q["crop"], cv2.COLOR_RGB2BGR))
        except Exception as e:
            print(f"  WARN: could not save crop {crop_name}: {e}")

        try:
            result = analyze_image(llm, q["crop"], crop_name)
        except Exception as e:
            result = {"file": crop_name, "raw": "", "parsed": None, "error": str(e), "attempts": 0}

        q_text = ""
        q_answer = ""
        q_status = ""
        if result["parsed"]:
            q_text = result["parsed"].get("question_text", "").strip()
            q_answer = result["parsed"].get("answer", "").strip()
            attempts = result.get("attempts", 1)
            if result.get("error"):
                q_status = f" (retries={attempts}, {result['error']})"
        elif result["error"]:
            q_text = f"[PARSE ERROR: {result['error']}]"
            q_status = f" (attempts={result.get('attempts', 1)})"

        entry = {
            "question_number": q_idx,
            "question_text": q_text,
            "answer": q_answer,
        }
        worksheet_qa["questions"].append(entry)
        display_text = q_text[:60] if q_text else '[empty]'
        answer_text = q_answer[:40] if q_answer else '[empty]'
        print(f"  Q{q_idx}: {display_text} -> {answer_text}{q_status}")

    all_results.append(worksheet_qa)

    # Save per-worksheet clean Q&A
    out_path = os.path.join(ws_dir, f"{stem}_qa.json")
    with open(out_path, "w") as f:
        json.dump(worksheet_qa, f, indent=2)
    print(f"  Saved: {out_path}")

# â”€â”€ Final summary â”€â”€
if image_paths:
    total_qs = sum(len(r["questions"]) for r in all_results)
    ok = sum(1 for r in all_results for q in r["questions"] if q.get("answer"))
    fail = sum(1 for r in all_results for q in r["questions"] if not q.get("answer"))
    print(f"\n{'='*60}")
    print(f"DONE: {len(all_results)} worksheet(s), {total_qs} total questions")
    print(f"  {ok} with answers, {fail} missing answers")
    print(f"{'='*60}")

    # Save combined Q&A
    combined = {"worksheets": all_results}
    combined_path = os.path.join(OUTPUT_DIR, "all_questions_answers.json")
    with open(combined_path, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"Combined Q&A: {combined_path}")


In [ ]:
!zip -r ./FLN_Results.zip ./FLN_Results